# 009_sam2_zeroshot_eval — Data Ingestion Pipeline

This notebook runs on **CPU**. Ensures polyp data is in Gold for SAM2 zero-shot evaluation.


In [ ]:
# Cell 1: Colab Environment Setup
from google.colab import drive, runtime
drive.mount("/content/drive")

# Clone code repos from GitHub
!git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning
!git clone --depth 1 -q https://github.com/khangnghiem/data-ingestion.git /content/data-ingestion

import os, sys
sys.path.insert(0, "/content/deep-learning")
sys.path.insert(0, "/content/data-ingestion")
os.chdir("/content/deep-learning/experiments/009_sam2_zeroshot_eval")


In [ ]:
import yaml

from transforms.medical.polyp_silver import process_polyp_dataset
from transforms.gold.segmentation_gold import prepare_gold_segmentation

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

experiment_name = config["experiment"]["name"]
data_cfg = config["data"]
print(f"Experiment: {experiment_name}")


In [ ]:
# Ensure Silver exists for eval datasets
silver_datasets = []
for source in data_cfg.get("sources", []):
    ds_name = source["name"]
    try:
        process_polyp_dataset(ds_name, "images", "masks")
    except Exception as e:
        print(f"Silver for {ds_name}: {e}")
    silver_datasets.append(ds_name)

gold_cfg = data_cfg.get("gold", {})
prepare_gold_segmentation(
    experiment_name=experiment_name,
    silver_datasets=silver_datasets,
    resolution=tuple(gold_cfg.get("resolution", [352, 352])),
)
print("✅ Gold data creation complete!")


In [ ]:
# Release Colab runtime
from google.colab import runtime
runtime.unassign()
